In [3]:
import pandas as pd
import numpy as np

# Load data
d2 = pd.read_csv("forceCoeff/x2.dat", sep=r"\s+")
d3 = pd.read_csv("forceCoeff/x3.dat", sep=r"\s+")
d5 = pd.read_csv("forceCoeff/x5.dat", sep=r"\s+")

# Extract signals (discard transient region)
y2 = d2["Cd"][50:]
y3 = d3["Cd"][50:]
y5 = d5["Cd"][50:]

# Step 1: Representative values (time-averaged Cd)
phi1 = np.mean(y2)  # coarse
phi2 = np.mean(y3)  # fine
phi3 = np.mean(y5)  # extra fine

# Step 2: Grid sizes and effective grid spacing
N1, N2, N3 = 2300, 5748, 32500

h1 = N1**(-1/3)
h2 = N2**(-1/3)
h3 = N3**(-1/3)

r21 = h1 / h2
r32 = h2 / h3

# Step 3: Apparent order (robust for oscillatory convergence)
ratio = (phi1 - phi2) / (phi2 - phi3)

p = np.log(abs(ratio)) / np.log(r21)

# Step 4: Richardson extrapolation (sign-aware)
s = np.sign(ratio)
phi_ext = phi3 + s * (phi3 - phi2) / (r32**p - 1)

# Step 5: Relative errors
e32 = abs((phi3 - phi2) / phi3)
e21 = abs((phi2 - phi1) / phi2)

# Step 6: GCI
Fs = 1.25  # safety factor
GCI32 = Fs * e32 / (r32**p - 1)
GCI21 = Fs * e21 / (r21**p - 1)

# Step 7: Asymptotic range check
AR = GCI21 / (GCI32 * r21**p)

# Output results
print(f"phi1 (coarse)      = {phi1:.6f}")
print(f"phi2 (fine)        = {phi2:.6f}")
print(f"phi3 (extra fine)  = {phi3:.6f}")
print()
print(f"r21                = {r21:.4f}")
print(f"r32                = {r32:.4f}")
print()
print(f"p (order)          = {p:.4f}")
print(f"phi_ext            = {phi_ext:.6f}")
print()
print(f"GCI32 (fine grid)  = {GCI32*100:.3f} %")
print(f"GCI21              = {GCI21*100:.3f} %")
print(f"Asymptotic ratio   = {AR:.3f}")

phi1 (coarse)      = 1.282411
phi2 (fine)        = 1.327163
phi3 (extra fine)  = 1.311905

r21                = 1.3571
r32                = 1.7815

p (order)          = 3.5243
phi_ext            = 1.314198

GCI32 (fine grid)  = 0.219 %
GCI21              = 2.181 %
Asymptotic ratio   = 3.402
